# 02 – Analyze: First look at July 2025
Goal: understand the data structure, validate key fields and explore delays by airline and route before loading all 12 months.

In [1]:
import pandas as pd

julio = pd.read_parquet("data/interim/ontime_2025_07.parquet")
julio.shape

(631428, 31)

In [2]:
julio.head()

,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,OriginCityName,...,ArrDel15,Cancelled,CancellationCode,Diverted,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2025,7,23,3,2025-07-23,AA,N450AN,969,MIA,"Miami, FL",...,0.0,0.0,<NA>,0.0,650.0,NaN,NaN,NaN,NaN,NaN
1,2025,7,24,4,2025-07-24,AA,N431AN,969,MIA,"Miami, FL",...,1.0,0.0,<NA>,0.0,650.0,0.0,0.0,0.0,0.0,74.0
2,2025,7,25,5,2025-07-25,AA,N444UW,969,MIA,"Miami, FL",...,0.0,0.0,<NA>,0.0,650.0,NaN,NaN,NaN,NaN,NaN
3,2025,7,26,6,2025-07-26,AA,N462AA,969,MIA,"Miami, FL",...,0.0,0.0,<NA>,0.0,650.0,NaN,NaN,NaN,NaN,NaN
4,2025,7,27,7,2025-07-27,AA,N458AL,969,MIA,"Miami, FL",...,0.0,0.0,<NA>,0.0,650.0,NaN,NaN,NaN,NaN,NaN


In [3]:
julio.head(1).T

,0
Year,2025
Month,7
DayofMonth,23
DayOfWeek,3
FlightDate,2025-07-23
Reporting_Airline,AA
Tail_Number,N450AN
Flight_Number_Reporting_Airline,969
Origin,MIA
OriginCityName,"Miami, FL"


In [4]:
julio[julio["ArrDel15"] == 1].head(1).T

,1
Year,2025
Month,7
DayofMonth,24
DayOfWeek,4
FlightDate,2025-07-24
Reporting_Airline,AA
Tail_Number,N431AN
Flight_Number_Reporting_Airline,969
Origin,MIA
OriginCityName,"Miami, FL"


In [5]:
tarde = julio[julio["ArrDel15"] == 1]
causas = ["CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"]

suma_causas = tarde[causas].sum(axis=1)
cuadra = suma_causas == tarde["ArrDelay"]

print(f"Vuelos que llegaron tarde: {len(tarde):,}")
print(f"Cuadran (suma = retraso):  {cuadra.sum():,}")
print(f"No cuadran:                {(~cuadra).sum():,}")

Vuelos que llegaron tarde: 177,012
Cuadran (suma = retraso):  177,012
No cuadran:                0


### Control test 1 — Delay causes reconcile to arrival delay
**Rule:** DOT Directive #40, sec. III.5 — cause minutes must equal arrival delay.
**Result:** 177,012 of 177,012 late flights reconcile (100%). ✅

In [6]:
operados = julio[(julio["Cancelled"] == 0) & (julio["Diverted"] == 0)]

por_aerolinea = operados.groupby("Reporting_Airline").agg(
    vuelos=("ArrDel15", "size"),
    tarde=("ArrDel15", "sum")
)
por_aerolinea["pct_tarde"] = (por_aerolinea["tarde"] / por_aerolinea["vuelos"] * 100).round(2)
por_aerolinea.sort_values("pct_tarde", ascending=False)

,vuelos,tarde,pct_tarde
Reporting_Airline,,,
F9,15893,5722.0,36.00
AA,84378,29516.0,34.98
B6,19512,6580.0,33.72
OH,19614,6064.0,30.92
AS,23012,7051.0,30.64
UA,68051,20682.0,30.39
G4,13855,4171.0,30.10
WN,123527,35731.0,28.93
YX,27563,7515.0,27.26


### Finding 1 — Late-arrival rate by airline (July 2025, all causes)
- Base: 612,811 operated flights (cancelled and diverted excluded); 71.1% arrived on time. Late flights reconcile to control test 1 (177,012).
- Among the 5 client airlines, late rates range from **24.84% (DL)** to **34.98% (AA)** — a 10.1-point gap.
- **WN** has the most late flights (35,731) because it is the largest carrier, but its rate (28.93%) is mid-range: counts measure size, rates measure performance.
- AA's regional affiliates OH (30.92%) and MQ (26.36%) performed better than AA mainline.
- ⚠️ One month, all causes combined (weather, NAS, carrier). Not yet a controllable-delay conclusion.

In [7]:
clientes = ["AA", "DL", "UA", "B6", "WN"]
tarde_clientes = operados[(operados["ArrDel15"] == 1) & (operados["Reporting_Airline"].isin(clientes))]

rutas = (tarde_clientes
         .groupby(["Reporting_Airline", "Origin", "Dest"])
         .size()
         .reset_index(name="vuelos_tarde")
         .sort_values("vuelos_tarde", ascending=False))
rutas.head(15)

,Reporting_Airline,Origin,Dest,vuelos_tarde
1294,DL,ATL,MCO,204
269,AA,DFW,LAX,183
1344,DL,ATL,TPA,181
2569,UA,LGA,ORD,174
274,AA,DFW,MCO,171
121,AA,CLT,MCO,169
2671,UA,ORD,LGA,163
2655,UA,ORD,EWR,162
1255,DL,ATL,DCA,157
278,AA,DFW,MIA,153


### Finding 2 — Routes with the most late flights (client airlines)
- Top routes by count depart from airline **hubs**: ATL (DL); DFW, CLT (AA); ORD, EWR, DEN, IAH (UA). Florida leisure destinations (MCO, TPA) recur.
- WN and B6 do not appear: their networks are less hub-concentrated.
- ⚠️ Counts favor high-frequency routes; rates are needed to identify poor performers.

**Hypothesis:** delays propagate through aircraft rotation at hubs — a late aircraft on one route delays its next route (late-aircraft effect).
**Next tests:** (1) late rate by route, (2) cause mix by route, (3) follow aircraft by tail number through the day.

In [8]:
clientes = ["AA", "DL", "UA", "B6", "WN"]
base = operados[operados["Reporting_Airline"].isin(clientes)]

rutas_tasa = base.groupby(["Reporting_Airline", "Origin", "Dest"]).agg(
    vuelos=("ArrDel15", "size"),
    tarde=("ArrDel15", "sum")
)
rutas_tasa["pct_tarde"] = (rutas_tasa["tarde"] / rutas_tasa["vuelos"] * 100).round(1)

print(f"Rutas totales de los 5 clientes: {len(rutas_tasa):,}")
print(f"Rutas con 100 o más vuelos:      {(rutas_tasa['vuelos'] >= 100).sum():,}")

rutas_tasa[rutas_tasa["vuelos"] >= 100].sort_values("pct_tarde", ascending=False).head(15)

Rutas totales de los 5 clientes: 4,625
Rutas con 100 o más vuelos:      1,380


vuelos  tarde  pct_tarde
Reporting_Airline Origin Dest                          
B6                MCO    EWR      103   61.0       59.2
AA                DFW    MCI      173  102.0       59.0
                         MCO      292  171.0       58.6
                         MSY      174  100.0       57.5
                         FAT      149   83.0       55.7
UA                JAC    DEN      121   67.0       55.4
AA                DFW    BNA      265  145.0       54.7
                         DEN      264  144.0       54.5
B6                MCO    JFK      224  122.0       54.5
UA                EWR    CLE      114   62.0       54.4
AA                CLT    LGA      234  127.0       54.3
WN                DEN    SFO      137   74.0       54.0
AA                DFW    SLC      187  101.0       54.0
                         ONT      193  104.0       53.9
                  ORD    MCO      176   94.0       53.4

### Finding 3 — Worst routes by late rate (routes with 100+ flights)
- **8 of the 15 worst routes are AA departures from DFW**, with late rates of 53.9%–59.0% vs. AA's July average of 34.98%. Destinations vary widely; the common factor is the origin hub.
- AA DFW→MCO (292 flights, 58.6%) and DFW→BNA (265 flights, 54.7%) rank high in **both** count and rate.
- No DL routes appear: DL's ATL routes led by count due to volume, not poor performance.

**Hypothesis status:** strengthened for AA at DFW, not yet proven. July thunderstorms could also explain it. Next: cause mix by route (test 2).

In [9]:
causas = ["CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"]

aa       = base[base["Reporting_Airline"] == "AA"]
aa_dfw   = aa[aa["Origin"] == "DFW"]
aa_resto = aa[aa["Origin"] != "DFW"]
otros    = base[base["Reporting_Airline"] != "AA"]

grupos = {"AA desde DFW": aa_dfw, "AA otros aeropuertos": aa_resto, "Otras 4 aerolíneas": otros}

for nombre, df in grupos.items():
    print(f"{nombre:22} | vuelos: {len(df):>7,} | % tarde: {df['ArrDel15'].mean():.1%}")

def mezcla(df):
    minutos = df[causas].sum()
    return (minutos / minutos.sum() * 100).round(1)

pd.DataFrame({nombre: mezcla(df) for nombre, df in grupos.items()})

AA desde DFW           | vuelos:  14,022 | % tarde: 44.7%
AA otros aeropuertos   | vuelos:  70,356 | % tarde: 33.0%
Otras 4 aerolíneas     | vuelos: 303,335 | % tarde: 28.3%


,AA desde DFW,AA otros aeropuertos,Otras 4 aerolíneas
CarrierDelay,31.8,30.8,29.6
WeatherDelay,2.7,7.3,5.6
NASDelay,12.5,13.8,22.5
SecurityDelay,0.1,0.1,0.1
LateAircraftDelay,52.9,48.1,42.2


In [10]:
filas = []
for aerolinea in clientes:
    df = base[base["Reporting_Airline"] == aerolinea]
    principal = df["Origin"].value_counts().idxmax()

    for etiqueta, sub in [(f"{aerolinea} desde {principal}", df[df["Origin"] == principal]),
                          (f"{aerolinea} otros aeropuertos", df[df["Origin"] != principal])]:
        fila = mezcla(sub)
        fila["vuelos"] = len(sub)
        fila["pct_tarde"] = round(sub["ArrDel15"].mean() * 100, 1)
        fila["min_por_vuelo"] = round(sub[causas].sum().sum() / len(sub), 1)
        fila.name = etiqueta
        filas.append(fila)

tabla = pd.DataFrame(filas)[["vuelos", "pct_tarde", "min_por_vuelo"] + causas]

promedio_tarde = base["ArrDel15"].mean() * 100
promedio_late = mezcla(base)["LateAircraftDelay"]
tabla["desv_pct_tarde"] = (tabla["pct_tarde"] - promedio_tarde).round(1)
tabla["desv_late_aircraft"] = (tabla["LateAircraftDelay"] - promedio_late).round(1)

print(f"Promedio 5 clientes → % tarde: {promedio_tarde:.1f}% | % LateAircraft: {promedio_late}%")
tabla

Promedio 5 clientes → % tarde: 29.8% | % LateAircraft: 44.2%


,vuelos,pct_tarde,min_por_vuelo,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,desv_pct_tarde,desv_late_aircraft
AA desde DFW,14022.0,44.7,35.0,31.8,2.7,12.5,0.1,52.9,14.9,8.7
AA otros aeropuertos,70356.0,33.0,32.7,30.8,7.3,13.8,0.1,48.1,3.2,3.9
DL desde ATL,21492.0,30.2,21.0,44.2,8.4,23.3,0.0,24.1,0.4,-20.1
DL otros aeropuertos,70753.0,23.2,19.2,39.8,4.6,20.6,0.0,35.0,-6.6,-9.2
UA desde DEN,9487.0,37.6,29.5,23.6,3.1,17.5,0.0,55.7,7.8,11.5
UA otros aeropuertos,58564.0,29.2,26.1,24.6,9.1,29.9,0.0,36.4,-0.6,-7.8
B6 desde BOS,3264.0,32.5,27.7,26.3,2.1,27.5,0.1,43.9,2.7,-0.3
B6 otros aeropuertos,16248.0,34.0,32.6,28.0,3.6,27.4,0.1,40.8,4.2,-3.4
WN desde DEN,8537.0,42.7,29.3,29.1,1.4,11.4,0.0,58.1,12.9,13.9
WN otros aeropuertos,114990.0,27.9,18.4,24.9,4.6,18.7,0.2,51.6,-1.9,7.4


In [11]:
principales = {aer: base[base["Reporting_Airline"] == aer]["Origin"].value_counts().idxmax()
               for aer in clientes}
print("Aeropuerto principal de cada cliente:", principales)

resultados = []
for apt in sorted(set(principales.values())):
    en_apt = base[base["Origin"] == apt]
    for aer in clientes:
        sub = en_apt[en_apt["Reporting_Airline"] == aer]
        if len(sub) == 0:
            continue
        pares = en_apt[en_apt["Reporting_Airline"] != aer]
        resultados.append({
            "aeropuerto": apt,
            "aerolinea": aer,
            "vuelos": len(sub),
            "pct_tarde": round(sub["ArrDel15"].mean() * 100, 1),
            "pct_tarde_pares": round(pares["ArrDel15"].mean() * 100, 1),
            "min_por_vuelo": round(sub[causas].sum().sum() / len(sub), 1),
            "pct_carrier": mezcla(sub)["CarrierDelay"],
            "pct_late_aircraft": mezcla(sub)["LateAircraftDelay"],
        })

comp = pd.DataFrame(resultados)
comp["brecha_vs_pares"] = (comp["pct_tarde"] - comp["pct_tarde_pares"]).round(1)
comp["confiable"] = comp["vuelos"] >= 100
comp.set_index(["aeropuerto", "aerolinea"])

Aeropuerto principal de cada cliente: {'AA': 'DFW', 'DL': 'ATL', 'UA': 'DEN', 'B6': 'BOS', 'WN': 'DEN'}


vuelos  pct_tarde  pct_tarde_pares  min_por_vuelo  \
aeropuerto aerolinea                                                      
ATL        AA            687       35.8             30.3           38.1   
           DL          21492       30.2             32.0           21.0   
           UA            622       27.5             30.5           30.4   
           B6            170       39.4             30.4           51.1   
           WN           1629       31.4             30.4           22.3   
BOS        AA           1380       27.2             30.0           25.9   
           DL           2291       26.9             30.5           18.8   
           UA           1127       28.1             29.8           26.6   
           B6           3264       32.5             27.8           27.7   
           WN            695       31.8             29.4           26.2   
DEN        AA            768       34.1             39.3           37.4   
           DL           1083       27.4             39.8           24.9   
           UA           9487       37.6             40.5           29.5   
           B6            137       43.8             39.1           30.4   
           WN           8537       42.7             36.4           29.3   
DFW        AA          14022       44.7             26.4           35.0   
           DL           1018       22.3             44.0           22.0   
           UA            772       30.6             43.3           33.2   
           B6             29       62.1             42.6          121.3   

                      pct_carrier  pct_late_aircraft  brecha_vs_pares  \
aeropuerto aerolinea                                                    
ATL        AA                25.3               58.9              5.5   
           DL                44.2               24.1             -1.8   
           UA                16.5               32.1             -3.0   
           B6                41.1               40.2              9.0   
           WN                22.4               47.2              1.0   
BOS        AA                29.9               42.8             -2.8   
           DL                33.4               37.0             -3.6   
           UA                15.5               38.9             -1.7   
           B6                26.3               43.9              4.7   
           WN                16.2               48.0              2.4   
DEN        AA                32.3               52.8             -5.2   
           DL                40.3               44.7            -12.4   
           UA                23.6               55.7             -2.9   
           B6                16.5               48.1              4.7   
           WN                29.1               58.1              6.3   
DFW        AA                31.8               52.9             18.3   
           DL                38.3               35.0            -21.7   
           UA                22.1               35.8            -12.7   
           B6                63.3               23.0             19.5   

                      confiable  
aeropuerto aerolinea             
ATL        AA              True  
           DL              True  
           UA              True  
           B6              True  
           WN              True  
BOS        AA              True  
           DL              True  
           UA              True  
           B6              True  
           WN              True  
DEN        AA              True  
           DL              True  
           UA              True  
           B6              True  
           WN              True  
DFW        AA              True  
           DL              True  
           UA              True  
           B6             False

In [12]:
comp["exceso_vuelos_tarde"] = (comp["vuelos"] * comp["brecha_vs_pares"] / 100).round(0)

comp[comp["confiable"]].sort_values("exceso_vuelos_tarde", ascending=False)[
    ["aeropuerto", "aerolinea", "vuelos", "pct_tarde", "pct_tarde_pares", "brecha_vs_pares", "exceso_vuelos_tarde"]
]

,aeropuerto,aerolinea,vuelos,pct_tarde,pct_tarde_pares,brecha_vs_pares,exceso_vuelos_tarde
15,DFW,AA,14022,44.7,26.4,18.3,2566.0
14,DEN,WN,8537,42.7,36.4,6.3,538.0
8,BOS,B6,3264,32.5,27.8,4.7,153.0
0,ATL,AA,687,35.8,30.3,5.5,38.0
9,BOS,WN,695,31.8,29.4,2.4,17.0
4,ATL,WN,1629,31.4,30.4,1.0,16.0
3,ATL,B6,170,39.4,30.4,9.0,15.0
13,DEN,B6,137,43.8,39.1,4.7,6.0
2,ATL,UA,622,27.5,30.5,-3.0,-19.0
7,BOS,UA,1127,28.1,29.8,-1.7,-19.0


### Finding 4 — Same-airport peer benchmark (July 2025, all causes)
Each client was compared against the other clients **departing from the same airport** (same weather, runways and ATC), using the pooled peer average. Only groups with 100+ flights are shown.

**Excess late flights = flights × gap vs. peers.** Top cases:
1. **AA at DFW:** 44.7% late vs. 26.4% for peers (+18.3 pts) → **≈2,566 excess late flights**, by far the largest.
2. **WN at DEN:** +6.3 pts → ≈538.
3. **B6 at BOS:** +4.7 pts → ≈153.

- The top 3 cases are each airline **at its own busiest airport**, consistent with the delay-propagation hypothesis.
- **DL** beats its peers at all four airports, including its hub ATL.
- **B6** is above peers at every airport, but its volumes are small.
- ⚠️ All causes combined. Next: isolate the controllable share.

In [13]:
dfw = julio[(julio["Origin"] == "DFW") & (julio["Reporting_Airline"].isin(clientes))]

filas = []
for aer, g in dfw.groupby("Reporting_Airline"):
    op = g[(g["Cancelled"] == 0) & (g["Diverted"] == 0)]
    cancel_a = int((g["CancellationCode"] == "A").sum())
    filas.append({
        "aerolinea": aer,
        "programados": len(g),
        "cancel_A": cancel_a,
        "pct_cancel_A": round(cancel_a / len(g) * 100, 2),
        "operados": len(op),
        "min_carrier_por_vuelo": round(op["CarrierDelay"].sum() / len(op), 1),
        "pct_vuelos_retraso_controlable": round(((op["ArrDel15"] == 1) & (op["CarrierDelay"] > 0)).mean() * 100, 1),
        "min_late_aircraft_por_vuelo": round(op["LateAircraftDelay"].sum() / len(op), 1),
    })

ctrl = pd.DataFrame(filas).set_index("aerolinea")
ctrl[ctrl["programados"] >= 100]

,programados,cancel_A,pct_cancel_A,operados,min_carrier_por_vuelo,pct_vuelos_retraso_controlable,min_late_aircraft_por_vuelo
aerolinea,,,,,,,
AA,14621,43,0.29,14022,11.1,33.7,18.5
DL,1054,11,1.04,1018,8.4,10.8,7.7
UA,801,6,0.75,772,7.3,13.2,11.9


### Finding 5 — Controllable disruption at DFW (July 2025)
| Metric | AA | DL | UA |
|---|---|---|---|
| Carrier cancellations (code A) | 0.29% | 1.04% | 0.75% |
| Carrier delay minutes per flight | 11.1 | 8.4 | 7.3 |
| Flights late with carrier-caused minutes | 33.7% | 10.8% | 13.2% |
| Late-aircraft minutes per flight (reported separately) | 18.5 | 7.7 | 11.9 |

- AA's gap at DFW has a **significant controllable component**: about 3x the share of flights with carrier-caused delay vs. DL.
- AA also shows about 2.4x DL's late-aircraft minutes (propagation).
- AA has the **lowest** carrier-cancellation rate. Open question for the client: is AA choosing to operate late rather than cancel? Not provable with this data.

In [14]:
import glob

cols = ["Year", "Month", "Reporting_Airline", "Origin", "ArrDel15", "Cancelled",
        "Diverted", "CancellationCode", "CarrierDelay", "LateAircraftDelay"]
archivos = sorted(glob.glob("data/interim/ontime_*.parquet"))

anio = pd.concat(
    [pd.read_parquet(f, columns=cols, filters=[("Reporting_Airline", "in", clientes)]) for f in archivos],
    ignore_index=True
)

print(f"Archivos leídos: {len(archivos)}")
print(f"Vuelos de los 5 clientes (12 meses): {len(anio):,}")
anio.groupby(["Year", "Month"]).size()

Archivos leídos: 12
Vuelos de los 5 clientes (12 meses): 4,471,010


Year  Month
2025  7        398073
      8        377006
      9        356307
      10       386984
      11       363748
      12       369498
2026  1        341862
      2        325446
      3        385932
      4        379568
      5        395347
      6        391239
dtype: int64

In [15]:
op12 = anio[(anio["Cancelled"] == 0) & (anio["Diverted"] == 0)].copy()
op12["periodo"] = op12["Year"].astype(str) + "-" + op12["Month"].astype(str).str.zfill(2)

casos = [("AA", "DFW"), ("DL", "ATL"), ("UA", "DEN"), ("WN", "DEN"), ("B6", "BOS")]
filas = []
for aer, apt in casos:
    en_apt = op12[op12["Origin"] == apt]
    for per, g in en_apt.groupby("periodo"):
        propia = g[g["Reporting_Airline"] == aer]
        pares = g[g["Reporting_Airline"] != aer]
        filas.append({
            "caso": f"{aer} en {apt}",
            "periodo": per,
            "brecha": round((propia["ArrDel15"].mean() - pares["ArrDel15"].mean()) * 100, 1),
        })

brechas = pd.DataFrame(filas).pivot(index="periodo", columns="caso", values="brecha")
print("Meses con brecha positiva (peor que sus pares):")
print((brechas > 0).sum())
brechas

Meses con brecha positiva (peor que sus pares):
caso
AA en DFW    12
B6 en BOS    10
DL en ATL     3
UA en DEN     0
WN en DEN    12
dtype: int64


caso,AA en DFW,B6 en BOS,DL en ATL,UA en DEN,WN en DEN
periodo,,,,,
2025-07,18.3,4.7,-1.8,-3.0,6.3
2025-08,20.2,1.8,-4.3,-3.5,6.5
2025-09,6.9,-8.4,-2.0,-5.6,8.5
2025-10,6.8,-5.3,-5.6,-10.8,12.8
2025-11,8.6,5.0,-3.6,-4.1,5.4
2025-12,13.6,12.0,1.4,-8.4,8.7
2026-01,16.1,13.9,4.2,-0.8,0.1
2026-02,12.9,16.7,1.2,-4.5,5.6
2026-03,10.0,10.1,-0.6,-7.2,9.5


### Finding 6 — 12-month confirmation (Jul 2025 – Jun 2026)
Same-airport gap vs. peers, each client at its busiest airport (4,471,010 client flights loaded; July results match the single-month analysis).

| Case | Months worse than peers | Avg. monthly gap | Pattern |
|---|---|---|---|
| AA at DFW | 12 / 12 | +12.5 pts | Structural — largest |
| WN at DEN | 12 / 12 | +9.0 pts | Structural, worsening in spring 2026 |
| B6 at BOS | 10 / 12 | +5.3 pts | Seasonal (winter) |
| DL at ATL | 3 / 12 | −2.7 pts | Better than peers except Dec–Feb |
| UA at DEN | 0 / 12 | −6.8 pts | Better than peers every month |

- **Denver is a natural experiment:** UA and WN share the same airport conditions, yet UA beat its peers every month and WN trailed them every month.
- The AA-at-DFW finding from July is **not a one-month anomaly**.